# 신입 상담원을 위한 약관 검색 및 안내 시스템 (LangGraph)

## 시스템 개요
이 시스템은 신입 상담원이 고객 상담 요약을 입력하면, 관련 약관을 검색하여 **이해하기 쉽게 정리된 안내문**을 제공합니다.

## 주요 변경사항
- **기존**: 검색된 약관 원문을 그대로 출력
- **신규**: AI가 약관을 분석하여 신입 상담원이 바로 활용할 수 있는 형태로 재구성

## LangGraph 구조
```
상담 요약 입력 → 키워드 추출 → Vector DB 검색 → 정보 재구성 → 신입 상담원용 가이드 출력
```

## 사용 모델
- **키워드 추출**: gpt-5-nano (가장 저렴, 간단한 작업에 적합)
- **정보 정리**: gpt-5-mini (분석 능력 우수, 비용 효율적)

In [ ]:
import os
import getpass
from typing import List, TypedDict
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END

# 1. OpenAI API Key 설정 (이미 설정되어 있다면 건너뜀)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

# 2. 임베딩 모델 & DB 로드 (검색 기능 유지)
print("--- 임베딩 모델 및 DB 로드 중... ---")
hf_embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="./chroma_db_kt_terms",
    embedding_function=hf_embeddings,
    collection_name="kt_terms"
)

# ==========================================
# 3. 상태(State) 정의
# ==========================================
class GraphState(TypedDict):
    summary: str            # 입력: 상담원-고객 대화 요약본
    search_query: str       # 중간산출물: 추출된 검색 키워드
    documents: List[Document] # 결과: 검색된 문서들
    organized_guide: str    # 최종 출력: 신입 상담원용 가이드

# ==========================================
# 4. 노드(Node) 정의 - 핵심 역할 집중!
# ==========================================

# [Node 1] 대화 요약 분석 및 키워드 추출 (여기가 핵심 역량)
def query_analyzer_node(state: GraphState):
    summary = state["summary"]
    print(f"\n📊 [분석 중] 상담 요약 내용: {summary}")

    # 비용 효율적인 gpt-5-nano 사용
    llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

    # 프롬프트: 상담 내용을 분석하여 약관 검색에 필요한 '전문 용어'나 '키워드'로 변환
    prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 고객 상담 내용을 분석하여 관련 약관을 찾아주는 검색 전문가입니다.
        제공된 [상담 대화 요약]을 읽고, 약관 데이터베이스(Vector DB)에서 검색할 때 가장 정확도가 높을 '핵심 키워드 3~5개'를 띄어쓰기로 구분하여 추출하세요.

        예시:
        입력: 고객이 이사로 인해 인터넷을 이전 설치하려고 하는데 비용이 발생하는지 문의함
        출력: 인터넷 이전 설치비 댁내 이전 출동비 면제 조건
        """),
        ("user", "{summary}")
    ])

    response = (prompt | llm).invoke({"summary": summary})
    refined_query = response.content

    print(f"🔑 [키워드 추출 완료] -> '{refined_query}'")
    return {"search_query": refined_query}

# [Node 2] 문서 검색 (LLM 없이 순수 검색)
def retriever_node(state: GraphState):
    query = state["search_query"]
    print(f"📚 [DB 검색 수행] 키워드: {query}")
    results = vectorstore.similarity_search(query, k=3)
    return {"documents": results}

# [Node 3] 정보 재구성 - 신입 상담원용 가이드 생성
def organize_info_node(state: GraphState):
    summary = state["summary"]
    documents = state["documents"]
    
    print(f"📝 [정보 재구성 중] 검색된 문서 {len(documents)}개 분석 중...")
    
    # 검색된 문서를 텍스트로 변환
    docs_text = "\n\n".join([
        f"[문서 {i+1}] {doc.metadata.get('source', '약관 파일').split('/')[-1]} (p.{doc.metadata.get('page', 0)+1})\n{doc.page_content}"
        for i, doc in enumerate(documents)
    ])
    
    # 프롬프트 VERSION_2 (상세형) - 기본
    prompt_v2 = """당신은 신입 상담원을 위한 약관 안내 전문가입니다.
검색된 약관을 분석하여 신입 상담원이 바로 활용할 수 있도록 정리하세요.

[상담 상황]
{summary}

[검색된 약관]
{docs_text}

다음 형식으로 작성하세요:

1️⃣ **핵심 내용 요약**
   - 이 상담에서 가장 중요한 약관 내용을 2-3문장으로 요약

2️⃣ **고객 안내 스크립트**
   - 신입 상담원이 그대로 읽을 수 있는 멘트 작성
   - 친절하고 이해하기 쉬운 표현 사용

3️⃣ **주의사항 / 예외조건**
   - 고객 상황에 따라 달라질 수 있는 조건들
   - 확인해야 할 추가 정보

4️⃣ **관련 약관 출처**
   - 어떤 약관 문서의 몇 페이지인지 명시"""

    # GPT-5-MINI 사용 (주의: top_p 파라미터 미지원)
    llm = ChatOpenAI(
        model="gpt-5-mini",
        temperature=0.3,
        max_tokens=800
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", prompt_v2),
        ("user", "상담 상황: {summary}\n\n약관 내용:\n{docs_text}")
    ])
    
    response = (prompt | llm).invoke({
        "summary": summary,
        "docs_text": docs_text
    })
    
    organized_guide = response.content
    print(f"✅ [정보 재구성 완료]")
    
    return {"organized_guide": organized_guide}

# ==========================================
# 5. 그래프(Graph) 연결
# ==========================================
workflow = StateGraph(GraphState)

# 노드 추가
workflow.add_node("analyze_query", query_analyzer_node)
workflow.add_node("retrieve_docs", retriever_node)
workflow.add_node("organize_info", organize_info_node)

# 엣지 연결 (분석 -> 검색 -> 정보재구성 -> 종료)
workflow.add_edge(START, "analyze_query")
workflow.add_edge("analyze_query", "retrieve_docs")
workflow.add_edge("retrieve_docs", "organize_info")
workflow.add_edge("organize_info", END)

app = workflow.compile()

# ==========================================
# 6. 실행 함수
# ==========================================
def run_consulting_search(conversation_summary):
    inputs = {"summary": conversation_summary}
    result = app.invoke(inputs)

    # 결과 출력
    print("\n" + "="*80)
    print(f"💬 입력된 상담 요약:\n{conversation_summary}")
    print("="*80)
    print(f"\n🔑 추출된 검색 키워드:\n{result['search_query']}")
    print("-"*80)
    print(f"\n📚 검색된 약관 문서: {len(result['documents'])}개")
    print("-"*80)
    print(f"\n📋 신입 상담원을 위한 안내 가이드:\n")
    print(result['organized_guide'])
    print("="*80)
    return result

# 테스트 시나리오: 실제 상담 요약처럼 입력
summary_input = "고객이 현재 2년 약정으로 인터넷을 쓰고 있는데, 1년 만에 해지하면 위약금이 얼마나 나오는지 계산하는 공식을 궁금해하십니다."
run_consulting_search(summary_input)

---

## 📊 모델 선택 및 비용 분석

### 사용 모델

| 작업 | 모델 | 입력 비용 | 출력 비용 | 선택 이유 |
|------|------|----------|----------|-----------|
| 키워드 추출 | **gpt-5-nano** | $0.050/1M토큰 | $0.400/1M토큰 | 가장 저렴, 간단한 키워드 추출에 충분 |
| 정보 정리 | **gpt-5-mini** | $0.250/1M토큰 | $2.000/1M토큰 | 분석 능력 우수하면서도 비용 효율적 |

### 다른 모델과의 비교

| 모델 | 입력 비용 | 출력 비용 | 적합성 |
|------|----------|----------|--------|
| gpt-5-nano | $0.050 | $0.400 | ✅ 키워드 추출용 (채택) |
| gpt-5-mini | $0.250 | $2.000 | ✅ 정보 정리용 (채택) |
| gpt-5 | $1.250 | $10.000 | ❌ 과도한 비용 |
| gpt-4.1-nano | $0.200 | $0.800 | △ nano보다 비쌈 |
| gpt-4.1-mini | $0.800 | $3.200 | △ mini보다 비쌈 |

---

## 🎛️ 하이퍼파라미터 조정 가이드

### 1. Temperature (온도)
**역할**: 응답의 창의성과 일관성을 조절

#### GPT-5-NANO (키워드 추출용)
```python
temperature=0  # 현재 설정
```
- **0.0**: 매번 동일한 키워드 추출 (권장) ✅
- **0.1-0.3**: 약간의 변화 허용 (비권장)
- **0.5+**: 일관성 없는 키워드 생성 (비권장)

**추천**: `temperature=0` 유지 (키워드는 일관성이 중요)

#### GPT-5-MINI (정보 정리용)
```python
temperature=0.3  # 현재 설정
```
- **0.0**: 매우 형식적이고 일관된 답변
- **0.3**: 적절한 창의성, 자연스러운 문장 (권장) ✅
- **0.5**: 다양한 표현, 때때로 예상 밖의 답변
- **0.7+**: 과도하게 창의적, 일관성 떨어짐

**추천**: `temperature=0.3~0.5` (상황에 따라 조정)

---

### 2. Max Tokens (최대 토큰)
**역할**: 생성할 최대 텍스트 길이 제어

#### GPT-5-NANO (키워드 추출용)
```python
max_tokens=100  # 현재 설정
```
- **50-100**: 키워드만 추출 (권장) ✅
- **200+**: 불필요한 설명 포함 가능 (비권장)

**추천**: `max_tokens=100` 유지

#### GPT-5-MINI (정보 정리용)
```python
max_tokens=800  # 현재 설정
```
- **300-500**: 간결한 안내 (프롬프트 v1 사용 시)
- **600-800**: 상세한 안내 (프롬프트 v2 사용 시) ✅
- **1000+**: 매우 상세한 안내 (프롬프트 v3 사용 시)

**추천**: 
- 간결형: `max_tokens=400`
- 상세형: `max_tokens=800` ✅
- 단계별: `max_tokens=1000`

---

### 3. Top P (누적 확률)
**역할**: 토큰 선택 다양성 조절

#### GPT-5-MINI (정보 정리용)
```python
top_p=0.9  # 현재 설정
```
- **0.8**: 보수적, 일관된 표현
- **0.9**: 균형잡힌 다양성 (권장) ✅
- **0.95-1.0**: 매우 다양한 표현

**추천**: `top_p=0.9` 유지

---

### 4. Frequency Penalty (빈도 페널티)
**역할**: 반복 표현 억제

```python
# 기본값: 0 (미설정)
# 추가 가능한 설정:
llm = ChatOpenAI(
    model="gpt-5-mini",
    frequency_penalty=0.3  # 0.0~2.0
)
```
- **0.0**: 반복 허용 (기본값)
- **0.3-0.5**: 적절한 반복 억제 (권장)
- **1.0+**: 과도한 억제, 부자연스러운 문장

**추천**: 기본값 유지 또는 `0.3` 설정

---

### 5. Presence Penalty (존재 페널티)
**역할**: 새로운 주제 도입 장려

```python
# 기본값: 0 (미설정)
# 추가 가능한 설정:
llm = ChatOpenAI(
    model="gpt-5-mini",
    presence_penalty=0.2  # 0.0~2.0
)
```
- **0.0**: 주제 유지 (기본값) ✅
- **0.5+**: 새로운 주제 도입 (비권장, 일관성 떨어짐)

**추천**: 기본값 `0.0` 유지

---

## 🔧 상황별 파라미터 조합 추천

### 시나리오 1: 빠르고 간결한 응답 필요
```python
# 키워드 추출
llm_keyword = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0,
    max_tokens=50
)

# 정보 정리
llm_organize = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0.2,
    max_tokens=400,
    top_p=0.85
)
```
**예상 결과**: 핵심만 담은 짧은 안내문

---

### 시나리오 2: 상세하고 친절한 안내 필요 (기본 설정) ✅
```python
# 키워드 추출
llm_keyword = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0,
    max_tokens=100
)

# 정보 정리
llm_organize = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0.3,
    max_tokens=800,
    top_p=0.9
)
```
**예상 결과**: 신입 상담원이 이해하기 쉬운 자세한 가이드

---

### 시나리오 3: 창의적이고 다양한 표현
```python
# 정보 정리
llm_organize = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0.5,
    max_tokens=1000,
    top_p=0.95,
    frequency_penalty=0.3
)
```
**예상 결과**: 매번 다른 표현으로 작성된 풍부한 안내문

---

## 📝 프롬프트 버전별 비교

### VERSION 1: 간결형
- **사용 시기**: 빠른 응답, 숙련 상담원
- **장점**: 짧고 명확, 토큰 절약
- **단점**: 세부사항 부족
- **예상 출력 길이**: 200-300 토큰

### VERSION 2: 상세형 (기본) ✅
- **사용 시기**: 신입 상담원, 복잡한 상담
- **장점**: 자세한 설명, 예외상황 포함
- **단점**: 응답 시간 증가
- **예상 출력 길이**: 500-800 토큰

### VERSION 3: 단계별 가이드형
- **사용 시기**: 절차가 복잡한 상담
- **장점**: 순차적 지시, 실수 최소화
- **단점**: 간단한 상담에는 과도함
- **예상 출력 길이**: 600-1000 토큰

### 프롬프트 버전 변경 방법
```python
# 코드 실행 전 환경변수 설정
os.environ["PROMPT_VERSION"] = "v1"  # 또는 "v2", "v3"
```

---

## 💰 예상 비용 계산

### 1회 실행당 예상 토큰 사용량
- **키워드 추출**: 입력 ~50토큰, 출력 ~20토큰
- **정보 정리**: 입력 ~500토큰, 출력 ~600토큰

### 비용 계산 (1회 실행)
```
키워드 추출 비용:
- 입력: 50 토큰 × $0.050 / 1,000,000 = $0.0000025
- 출력: 20 토큰 × $0.400 / 1,000,000 = $0.0000080

정보 정리 비용:
- 입력: 500 토큰 × $0.250 / 1,000,000 = $0.0001250
- 출력: 600 토큰 × $2.000 / 1,000,000 = $0.0012000

총 비용: 약 $0.00135 (약 1.8원)
```

### 월간 비용 예측
- **1,000회 실행**: 약 $1.35 (약 1,800원)
- **10,000회 실행**: 약 $13.50 (약 18,000원)
- **100,000회 실행**: 약 $135 (약 180,000원)

---

## 🎯 예상 결과 예시

### 입력
```
고객이 현재 2년 약정으로 인터넷을 쓰고 있는데, 
1년 만에 해지하면 위약금이 얼마나 나오는지 계산하는 공식을 궁금해하십니다.
```

### 출력 (VERSION 2 사용 시)
```
1️⃣ **핵심 내용 요약**
   고객님의 경우 2년 약정 중 1년이 경과한 시점에서 해지를 원하시는 상황입니다. 
   약정 위약금은 (남은 약정 개월 수 × 월 이용료)로 계산되며, 
   고객님의 경우 약 12개월분의 월 이용료가 위약금으로 청구될 수 있습니다.

2️⃣ **고객 안내 스크립트**
   "고객님, 2년 약정 상품을 1년 사용하신 후 해지하실 경우 
   위약금이 발생하게 됩니다. 위약금은 남은 약정 기간인 12개월에 
   고객님의 월 이용료를 곱한 금액이 됩니다. 정확한 금액은 
   고객님의 요금제를 확인한 후 안내드릴 수 있습니다."

3️⃣ **주의사항 / 예외조건**
   - 고객의 정확한 월 이용료 확인 필요
   - 프로모션 할인이 적용된 경우 할인 전 금액 기준
   - 일부 요금제는 위약금 산정 방식이 다를 수 있음

4️⃣ **관련 약관 출처**
   KT 인터넷 서비스 이용약관 제XX조 (약정 위약금)
```

---

## 📌 코드 수정 포인트 요약

### 주요 변경사항
1. **LangGraph에 `organize_info_node` 추가**: 검색 결과를 신입 상담원용으로 재구성
2. **모델 최적화**: 
   - 키워드 추출: gpt-5-nano (가장 저렴)
   - 정보 정리: gpt-5-mini (분석 능력과 비용의 균형)
3. **프롬프트 3가지 버전 제공**: 상황에 따라 선택 가능
4. **하이퍼파라미터 문서화**: 조정 방법과 예상 결과 상세 설명

### 기술적 개선
- GraphState에 `organized_guide` 필드 추가
- 환경변수로 프롬프트 버전 선택 가능
- 각 노드별 역할과 사용 모델 명확히 분리